# Inference: anonymize faces and license plates with a trained segmentation model

Pick a trained model, then see how well it does on held-out test images, what it marks on full images and close-ups,
how it behaves on unlabeled images of your own, and what any single image looks like anonymized.

Each whole image is resized so its longer side is the model input size, padded to a square and labelled in one pass,
exactly as `python -m inference.anonymize` does, so what is shown here is what anonymizing real images would produce.
Images of any sizes are labelled together, `BATCH_SIZE` per forward pass, with the script's own batching function.
Everything the model marks as face or plate with a probability above the threshold is blurred; the mask is grown by
a few pixels first, so the edges of a face or plate are blurred too. The growth and the blur strength scale with each
image's size, so a photo is blurred the same way at any resolution.

## 1. Settings

In [ ]:
WEIGHTS_DIR = "model_weights/v0_0_1_pp4av_seg3_mita_segmentation_tiny_test"   # model_weights/<model id>
TEST_PATHS = ["data/v0_0_1/pp4av_segmentation_test"]                       # merged, like dataset_paths in a config
UNLABELED_PATHS = []       # prepared unlabeled datasets, e.g. ["data/v0_0_1/dashcam_unlabeled"]
N_PER_LABEL = 3            # random test images per label in section 3, new ones on every run
N_CLOSEUPS = 100           # random objects per label in section 4, new ones on every run
CLOSEUP_COLS = 10          # objects per grid row in section 4; each takes two cells: original, anonymized
N_UNLABELED = 6            # random unlabeled images shown in section 5
DEVICE = "cuda"
EVAL_LIMIT = None          # images used for the test metrics (None = all)
BATCH_SIZE = 32            # images per forward pass, for the test metrics and every section below (script: 32)

THRESHOLD = 0.5            # P(private) above which a pixel is masked
MASK_DILATE_PX = 3         # grow the mask slightly before blurring    } in pixels of an image whose longer side is
BLUR_SIGMA = 8             # Gaussian blur strength                    } REFERENCE_SIZE; both scale with each image
REFERENCE_SIZE = 480

In [ ]:
import json, math, random, time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from inference.anonymize import anonymize as anonymize_with, private_probs as private_probs_with
from models.mita_segmentation import MitaSegmentation
from trainer.common import load_merged_dataset
from trainer.segmentation import class_lut, decode_row, evaluate, undecoded

model = MitaSegmentation.load(WEIGHTS_DIR, device=DEVICE)
state = json.loads((Path(WEIGHTS_DIR) / "train_state.json").read_text())
test_ds = load_merged_dataset(TEST_PATHS)
pd.Series({
    **{k: v for k, v in model.config.items()},
    "trained": f"epoch {state['epoch']}, step {state['step']}" if "epoch" in state else f"optuna trial {state['trial']}",
    "model input": f"{model.input_size}x{model.input_size} px: whole image, longer side resized to fit, padded",
    "test images": f"{len(test_ds)} from {sorted(set(test_ds['subset']))}",
}).to_frame("model")

## 2. Test metrics

Dice and IoU per class on the test images, counted over all their pixels: the same measure training used on its
validation images, so the two columns are directly comparable. A test score far below the validation score means
validation was easier than real use, for example because it shared near-duplicate frames with training.

In [ ]:
eval_ds = test_ds if EVAL_LIMIT is None else test_ds.select(range(min(EVAL_LIMIT, len(test_ds))))
t = time.perf_counter()
test_metrics = evaluate(model, eval_ds, batch_size=BATCH_SIZE)
print(f"{len(eval_ds)} images in {time.perf_counter() - t:.0f} s")
pd.DataFrame({"test": test_metrics, "valid (at training)": state.get("valid", {})}).round(4)

## 3. Each label: random test images

Every labelled object in the test images is found first: a connected region of one label's pixels in a mask, as the
data pipelines drew it (mask value v is the row's `label_names[v]`, translated to the model's classes). Then, for
each label, a few test images containing it are drawn at random, new ones every time the cell below is run: the
input with that label's boxes (green; other labels' boxes orange), the region the model blurs with the same boxes
on top, so a missed object shows as a green box over black, and the anonymized image.

In [ ]:
def private_probs(imgs_rgb):
    """A list of RGB uint8 images [H, W, 3] of any sizes, labelled BATCH_SIZE per forward pass -> for each, in order,
    P(not background) float32 [H, W], and the model time per image in seconds."""
    t = time.perf_counter()
    probs = list(private_probs_with(model, imgs_rgb, BATCH_SIZE))
    return probs, (time.perf_counter() - t) / max(len(imgs_rgb), 1)

def boxes(binary):
    """Connected regions of a 0/1 mask -> boxes x1, y1, x2, y2."""
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary.astype(np.uint8), connectivity=8)
    return [(int(x), int(y), int(x + w), int(y + h)) for x, y, w, h in stats[1:, :4]]

def label_and_anonymize(row_ids):
    """Test rows -> for each, in order: the RGB image, its class mask (indices into model.classes), the anonymized
    image and the blurred region. Decoded and labelled BATCH_SIZE at a time, so only one batch of images is held."""
    for start in range(0, len(row_ids), BATCH_SIZE):
        batch = [decode_row(raw[int(i)], model.classes) for i in row_ids[start:start + BATCH_SIZE]]
        for (img, label), prob in zip(batch, private_probs_with(model, [img for img, _ in batch], BATCH_SIZE)):
            yield (img, label) + anonymize(img, prob)

def anonymize(img_rgb, prob):
    return anonymize_with(img_rgb, prob, THRESHOLD, MASK_DILATE_PX, BLUR_SIGMA, REFERENCE_SIZE)

raw = undecoded(test_ds)
image_ids = test_ds["image_id"]
labels = model.classes[1:]                                   # model.classes[0] is the background
objects_by_label = {c: [] for c in labels}                   # label -> [(test row, box)]
for i, r in enumerate(raw.select_columns(["mask", "label_names"])):
    mask = cv2.imdecode(np.frombuffer(r["mask"]["bytes"], np.uint8), cv2.IMREAD_UNCHANGED)
    mask = class_lut(r["label_names"], model.classes)[mask]
    for k in np.unique(mask[mask > 0]):
        objects_by_label[model.classes[k]] += [(i, b) for b in boxes(mask == k)]

pd.DataFrame({c: {"images": len({i for i, _ in objs}), "objects": len(objs),
                  "median size (px)": np.median([max(x2 - x1, y2 - y1) for _, (x1, y1, x2, y2) in objs]) if objs else None}
              for c, objs in objects_by_label.items()}).T

In [ ]:
for k, c in enumerate(labels, start=1):
    rows_with = sorted({i for i, _ in objects_by_label[c]})
    if not rows_with:
        print(f"{c}: no objects in the test images")
        continue
    picks = random.sample(rows_with, min(N_PER_LABEL, len(rows_with)))      # unseeded: new images on every run
    for i, (img, label, anon, blurred) in zip(picks, label_and_anonymize(picks)):
        own = boxes(label == k)
        others = [b for j in range(1, len(model.classes)) if j != k for b in boxes(label == j)]
        fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
        axes[0].imshow(img)
        axes[1].imshow(blurred, cmap="gray", vmin=0, vmax=1)
        for ax in axes[:2]:
            for (x1, y1, x2, y2), color in [(b, "orange") for b in others] + [(b, "lime") for b in own]:
                ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, lw=1.5))
        axes[0].set_title(f"[{c}] {image_ids[i]}: {len(own)} {c} box(es)")
        axes[1].set_title(f"blurred region ({100 * blurred.mean():.2f}% of pixels) + labelled boxes")
        axes[2].imshow(anon); axes[2].set_title("anonymized")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()

## 4. Each label: close-ups of random objects, original vs. anonymized

For each label, `N_CLOSEUPS` labelled objects drawn at random from all test images, new ones on every run. Each
object takes two neighbouring cells, the original crop and the same crop anonymized, `CLOSEUP_COLS` objects per row
(10 x 20 cells for 100 objects). The crop is centred on the object and three times its size, and the size above
each original is the object's box in pixels of the full image. Each image is labelled whole, as in real use, and
only the crops are kept.

In [ ]:
def crop(arr, box):
    x1, y1, x2, y2 = box
    cx, cy, r = (x1 + x2) // 2, (y1 + y2) // 2, int(max(x2 - x1, y2 - y1) * 1.5 + 8)
    return arr[max(0, cy - r):cy + r, max(0, cx - r):cx + r]

for c in labels:
    pool = objects_by_label[c]
    if not pool:
        print(f"{c}: no objects in the test images")
        continue
    picks = random.sample(pool, min(N_CLOSEUPS, len(pool)))                 # unseeded: new objects on every run
    by_row = defaultdict(list)
    for i, box in picks:
        by_row[i].append(box)
    crops = []
    for i, (img, _, anon, _) in zip(by_row, label_and_anonymize(list(by_row))):
        crops += [(crop(img, b), crop(anon, b), b) for b in by_row[i]]

    n_rows = math.ceil(len(crops) / CLOSEUP_COLS)
    fig, axes = plt.subplots(n_rows, 2 * CLOSEUP_COLS, figsize=(0.9 * 2 * CLOSEUP_COLS, 1.05 * n_rows + 0.6),
                             squeeze=False, gridspec_kw=dict(wspace=0.05, hspace=0.3))
    for ax in axes.flat:
        ax.axis("off")
    for n, (orig, anon, (x1, y1, x2, y2)) in enumerate(crops):
        r, col = divmod(n, CLOSEUP_COLS)
        axes[r, 2 * col].imshow(orig, interpolation="nearest")
        axes[r, 2 * col].set_title(f"{x2 - x1}x{y2 - y1}", fontsize=6, pad=2)
        axes[r, 2 * col + 1].imshow(anon, interpolation="nearest")
        axes[r, 2 * col + 1].set_title("anon.", fontsize=6, pad=2)
    fig.suptitle(f"{c}: {len(crops)} random objects of {len(pool)} in the test images (original | anonymized)", y=1.0)
    plt.show()

## 5. Unlabeled images

Random images from the unlabeled datasets listed in the settings. Without labels there is nothing to score, so only
the prediction and the anonymized result are shown.

In [ ]:
if UNLABELED_PATHS:
    unlabeled = undecoded(load_merged_dataset(UNLABELED_PATHS).select_columns(["image_id", "image"]))
    picks = random.Random(0).sample(range(len(unlabeled)), min(N_UNLABELED, len(unlabeled)))
    rows = [unlabeled[i] for i in picks]
    imgs = [cv2.cvtColor(cv2.imdecode(np.frombuffer(r["image"]["bytes"], np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            for r in rows]
    probs, dt = private_probs(imgs)                              # any sizes, BATCH_SIZE per forward pass
    for row, img, prob in zip(rows, imgs, probs):
        anon, mask = anonymize(img, prob)
        fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
        axes[0].imshow(img); axes[0].set_title(f"{row['image_id']}  ({img.shape[1]}x{img.shape[0]})")
        axes[1].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[1].set_title(f"P(private)   ({dt * 1000:.0f} ms per image)")
        axes[2].imshow(anon); axes[2].set_title(f"anonymized ({100 * mask.mean():.2f}% of pixels)")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()

## 6. Anonymize image files

In [ ]:
IMAGE_PATHS = []           # e.g. ["/path/to/street.jpg", "/path/to/parking.png"]; any sizes, labelled together
OUTPUT_DIR = None          # e.g. "/path/to/out": each written there under its own file name; None = only show them

if IMAGE_PATHS:
    imgs = [cv2.imread(str(p), cv2.IMREAD_COLOR) for p in IMAGE_PATHS]       # orientation tag applied
    unreadable = [p for p, img in zip(IMAGE_PATHS, imgs) if img is None]
    if unreadable:
        raise FileNotFoundError(f"cannot read {unreadable}")
    imgs = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in imgs]
    probs, _ = private_probs(imgs)
    for path, img, prob in zip(IMAGE_PATHS, imgs, probs):
        anon, mask = anonymize(img, prob)
        if OUTPUT_DIR is not None:
            Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(Path(OUTPUT_DIR) / Path(path).name), cv2.cvtColor(anon, cv2.COLOR_RGB2BGR))
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        axes[0].imshow(img); axes[0].set_title(f"{Path(path).name}  ({img.shape[1]}x{img.shape[0]})")
        axes[1].imshow(anon); axes[1].set_title(f"anonymized ({100 * mask.mean():.2f}% of pixels)")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()